# 👁️ Göz ROI Çıkarma — Colab v4

Bu notebook, `deney1_frames/real|fake/train|val|test` yapısını okur ve çıktıyı aynı sırayla korur:

```text
eye_roi_output_v4/
├── real/
│   ├── train/
│   │   ├── left/
│   │   ├── right/
│   │   ├── combined/
│   │   ├── landmarks/
│   │   ├── debug/
│   │   ├── metadata.csv
│   │   └── statistics.json
│   ├── val/
│   └── test/
└── fake/
    ├── train/
    ├── val/
    └── test/
```

Bir frame içindeki bütün yüzler işlenir. Kapalı göz, yan bakış ve hafif baş dönüşü bu aşamada silinmez.


In [ ]:
# 1) Colab bağımlılıkları
!pip -q install mediapipe opencv-python-headless numpy pandas matplotlib


In [ ]:
# 2) Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 3) Face Landmarker modelini indir
from pathlib import Path
import urllib.request

MODEL_PATH = Path('/content/face_landmarker.task')
MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task'

if not MODEL_PATH.exists():
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print('Model hazır:', MODEL_PATH, '| Boyut:', MODEL_PATH.stat().st_size, 'bayt')


## 4) Giriş ve çıktı klasörleri

Giriş yapısı şu şekilde olmalıdır:

```text
deney1_frames/
├── real/
│   ├── train/
│   ├── val/
│   └── test/
└── fake/
    ├── train/
    ├── val/
    └── test/
```

Frame'ler doğrudan split klasöründe veya video alt klasörlerinde bulunabilir.


In [ ]:
# SADECE BU HÜCREDEKİ DRIVE YOLUNU KENDİ KLASÖRÜNE GÖRE DÜZENLE
from pathlib import Path

INPUT_ROOT = Path('/content/drive/MyDrive/AISC DeepFake Çalışmaları/deney1_frames')
OUTPUT_ROOT = Path('/content/drive/MyDrive/AISC DeepFake Çalışmaları/eye_roi_output_v4')

# İlk denemede 20-30 frame kullan. Debug doğruysa None yap.
SMOKE_TEST_LIMIT = 20

NUM_FACES = 10
DET_CONF = 0.30
PRESENCE_CONF = 0.30
TRACKING_CONF = 0.30
EYE_PADDING = 0.25
COMBINED_PADDING = 0.18
OUTPUT_SIZE = 224
FLUSH_EVERY = 25
OVERWRITE = False
SAVE_DEBUG = True

print('Input :', INPUT_ROOT)
print('Output:', OUTPUT_ROOT)


## 5) Ana kod

In [ ]:
# -*- coding: utf-8 -*-
"""
01_extract_eye_rois.py

Hazır frame görüntülerindeki TÜM yüzleri bulur ve her yüz için:
- sol göz ROI
- sağ göz ROI
- iki gözü birlikte içeren combined ROI
- debug overlay
- metadata.csv

üretir.

Önemli:
- Bir frame'de 1, 2, 3... yüz olabilir.
- Kapalı göz, yan bakış ve dönük kafa silinmez.
- Her yüz ayrı face_id ile işlenir.
- Sağ/sol göz, görüntüye bakan kişinin anatomik sağı/soludur.
- Kesintiden sonra metadata.csv üzerinden devam eder.
- CSV atomik yazılır.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import os
import re
import sys
import tempfile
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Sequence

import cv2
import mediapipe as mp
import numpy as np


# ---------------------------------------------------------------------
# MediaPipe Face Landmarker landmark indeksleri
# ---------------------------------------------------------------------
# MediaPipe indeksleri kişinin anatomik tarafına göre adlandırılmıştır.
# 33-133 bölgesi kişinin SAĞ gözü, 362-263 bölgesi kişinin SOL gözüdür.
RIGHT_EYE_CONTOUR = [
    33, 7, 163, 144, 145, 153, 154, 155,
    133, 173, 157, 158, 159, 160, 161, 246
]
RIGHT_IRIS = [468, 469, 470, 471, 472]

LEFT_EYE_CONTOUR = [
    362, 382, 381, 380, 374, 373, 390, 249,
    263, 466, 388, 387, 386, 385, 384, 398
]
LEFT_IRIS = [473, 474, 475, 476, 477]

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


@dataclass(frozen=True)
class Config:
    input_root: Path
    output_root: Path
    model_path: Path

    num_faces: int = 10
    min_face_detection_confidence: float = 0.30
    min_face_presence_confidence: float = 0.30
    min_tracking_confidence: float = 0.30

    eye_padding_ratio: float = 0.25
    combined_padding_ratio: float = 0.18
    output_size: int = 224

    save_left_right: bool = True
    save_combined: bool = True
    save_debug: bool = True

    flush_every: int = 50
    overwrite: bool = False


METADATA_FIELDS = [
    "sample_id",
    "source_frame",
    "relative_frame_path",
    "label",
    "split",
    "video_id",
    "frame_stem",
    "face_id",
    "face_id_scope",
    "faces_in_frame",

    "left_eye_detected",
    "right_eye_detected",
    "left_eye_path",
    "right_eye_path",
    "combined_eye_path",
    "debug_path",
    "landmarks_path",

    "left_bbox_x1",
    "left_bbox_y1",
    "left_bbox_x2",
    "left_bbox_y2",
    "right_bbox_x1",
    "right_bbox_y1",
    "right_bbox_x2",
    "right_bbox_y2",
    "combined_bbox_x1",
    "combined_bbox_y1",
    "combined_bbox_x2",
    "combined_bbox_y2",

    "left_eye_width_px",
    "left_eye_height_px",
    "right_eye_width_px",
    "right_eye_height_px",

    "left_iris_landmarks_available",
    "right_iris_landmarks_available",
    "left_iris_visible_estimate",
    "right_iris_visible_estimate",
    "landmark_count",

    "status",
    "error",
    "processing_ms",
]


def parse_args() -> Config:
    parser = argparse.ArgumentParser(
        description="Hazır frame'lerden çoklu yüz destekli göz ROI çıkarır."
    )
    parser.add_argument("--input-root", required=True, type=Path)
    parser.add_argument("--output-root", required=True, type=Path)
    parser.add_argument("--model-path", required=True, type=Path)

    parser.add_argument("--num-faces", type=int, default=10)
    parser.add_argument("--det-conf", type=float, default=0.30)
    parser.add_argument("--presence-conf", type=float, default=0.30)
    parser.add_argument("--tracking-conf", type=float, default=0.30)

    parser.add_argument("--eye-padding", type=float, default=0.25)
    parser.add_argument("--combined-padding", type=float, default=0.18)
    parser.add_argument("--output-size", type=int, default=224)
    parser.add_argument("--flush-every", type=int, default=50)

    parser.add_argument("--no-debug", action="store_true")
    parser.add_argument("--overwrite", action="store_true")

    args = parser.parse_args()

    return Config(
        input_root=args.input_root.resolve(),
        output_root=args.output_root.resolve(),
        model_path=args.model_path.resolve(),
        num_faces=args.num_faces,
        min_face_detection_confidence=args.det_conf,
        min_face_presence_confidence=args.presence_conf,
        min_tracking_confidence=args.tracking_conf,
        eye_padding_ratio=args.eye_padding,
        combined_padding_ratio=args.combined_padding,
        output_size=args.output_size,
        save_debug=not args.no_debug,
        flush_every=args.flush_every,
        overwrite=args.overwrite,
    )


def validate_config(cfg: Config) -> None:
    if not cfg.input_root.exists():
        raise FileNotFoundError(f"Frame klasörü bulunamadı: {cfg.input_root}")
    if not cfg.model_path.exists():
        raise FileNotFoundError(f"Face Landmarker modeli bulunamadı: {cfg.model_path}")
    if cfg.output_size <= 0:
        raise ValueError("output_size pozitif olmalıdır.")
    if cfg.num_faces <= 0:
        raise ValueError("num_faces en az 1 olmalıdır.")
    for name, value in {
        "det_conf": cfg.min_face_detection_confidence,
        "presence_conf": cfg.min_face_presence_confidence,
        "tracking_conf": cfg.min_tracking_confidence,
    }.items():
        if not 0.0 <= value <= 1.0:
            raise ValueError(f"{name} 0 ile 1 arasında olmalıdır.")


def iter_images(root: Path) -> list[Path]:
    images = [
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
    ]
    return sorted(images, key=lambda p: str(p).lower())


def infer_dataset_fields(frame_path: Path, input_root: Path) -> tuple[str, str, str]:
    """Label, split ve video_id bilgisini klasör yapısından güvenli çıkarır.

    Desteklenen örnekler:
      root/real/train/video_001/frame_0001.jpg
      root/train/real/video_001/frame_0001.jpg
      root/real/train/frame_video001_0001.jpg

    Video klasörü yoksa dosya adındaki son frame numarası temizlenerek video_id
    tahmin edilir. Böylece bütün görüntülerin yanlışlıkla ``train`` video_id'sini
    alması önlenir.
    """
    rel = frame_path.relative_to(input_root)
    parts = list(rel.parts)
    parts_lower = [part.lower() for part in parts]

    label = next((x for x in ("real", "fake") if x in parts_lower), "unknown")
    split = next(
        (x for x in ("train", "val", "validation", "test") if x in parts_lower),
        "unknown",
    )
    if split == "validation":
        split = "val"

    excluded = {"real", "fake", "train", "val", "validation", "test"}
    parent_candidates = [
        part for part in rel.parent.parts
        if part.lower() not in excluded and part not in {"", "."}
    ]

    if parent_candidates:
        video_id = parent_candidates[-1]
    else:
        stem = frame_path.stem
        # frame_000123, video01_frame_0042, video01_f0042 gibi son ekleri temizle.
        video_id = re.sub(
            r"(?i)(?:[_-]?(?:frame|frm|f))?[_-]?\d{1,8}$", "", stem
        ).strip("_- ")
        if not video_id:
            # Güvenli son çare: her frame'i yanlışlıkla aynı videoda toplama.
            video_id = f"unknown_video_{hashlib.sha1(rel.as_posix().encode('utf-8')).hexdigest()[:10]}"

    return label, split, video_id


def stable_sample_id(relative_path: str, face_id: int) -> str:
    raw = f"{relative_path}|face={face_id}".encode("utf-8")
    return hashlib.sha1(raw).hexdigest()[:16]


def landmarks_to_arrays(
    landmarks: Sequence,
    image_width: int,
    image_height: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Normalize (x,y,z) ve piksel (x,y) landmark dizilerini döndürür."""
    normalized = np.asarray(
        [
            (
                float(np.clip(lm.x, 0.0, 1.0)),
                float(np.clip(lm.y, 0.0, 1.0)),
                float(lm.z),
            )
            for lm in landmarks
        ],
        dtype=np.float32,
    )
    pixels = np.empty((len(normalized), 2), dtype=np.int32)
    pixels[:, 0] = np.rint(normalized[:, 0] * (image_width - 1)).astype(np.int32)
    pixels[:, 1] = np.rint(normalized[:, 1] * (image_height - 1)).astype(np.int32)
    return normalized, pixels


def valid_indices(points: np.ndarray, indices: Sequence[int]) -> list[int]:
    return [idx for idx in indices if 0 <= idx < len(points)]


def bbox_from_indices(
    points: np.ndarray,
    indices: Sequence[int],
    image_width: int,
    image_height: int,
    padding_ratio: float,
) -> tuple[int, int, int, int] | None:
    ids = valid_indices(points, indices)
    if not ids:
        return None

    selected = points[ids]
    x1, y1 = selected.min(axis=0)
    x2, y2 = selected.max(axis=0)

    width = max(1, int(x2 - x1))
    height = max(1, int(y2 - y1))
    pad_x = max(2, int(round(width * padding_ratio)))
    pad_y = max(2, int(round(height * padding_ratio)))

    x1 = max(0, int(x1 - pad_x))
    y1 = max(0, int(y1 - pad_y))
    x2 = min(image_width, int(x2 + pad_x + 1))
    y2 = min(image_height, int(y2 + pad_y + 1))

    if x2 - x1 < 4 or y2 - y1 < 4:
        return None
    return x1, y1, x2, y2


def union_bbox(
    bbox_a: tuple[int, int, int, int] | None,
    bbox_b: tuple[int, int, int, int] | None,
    image_width: int,
    image_height: int,
    padding_ratio: float,
) -> tuple[int, int, int, int] | None:
    boxes = [b for b in (bbox_a, bbox_b) if b is not None]
    if not boxes:
        return None

    x1 = min(b[0] for b in boxes)
    y1 = min(b[1] for b in boxes)
    x2 = max(b[2] for b in boxes)
    y2 = max(b[3] for b in boxes)

    width = max(1, x2 - x1)
    height = max(1, y2 - y1)
    pad_x = int(round(width * padding_ratio))
    pad_y = int(round(height * padding_ratio))

    return (
        max(0, x1 - pad_x),
        max(0, y1 - pad_y),
        min(image_width, x2 + pad_x),
        min(image_height, y2 + pad_y),
    )


def crop_and_letterbox(
    image_bgr: np.ndarray,
    bbox: tuple[int, int, int, int] | None,
    output_size: int,
) -> np.ndarray | None:
    if bbox is None:
        return None

    x1, y1, x2, y2 = bbox
    crop = image_bgr[y1:y2, x1:x2]
    if crop.size == 0:
        return None

    h, w = crop.shape[:2]
    scale = min(output_size / w, output_size / h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))

    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    resized = cv2.resize(crop, (new_w, new_h), interpolation=interpolation)

    canvas = np.zeros((output_size, output_size, 3), dtype=np.uint8)
    offset_x = (output_size - new_w) // 2
    offset_y = (output_size - new_h) // 2
    canvas[offset_y:offset_y + new_h, offset_x:offset_x + new_w] = resized
    return canvas


def atomic_imwrite(path: Path, image: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    suffix = path.suffix if path.suffix else ".jpg"
    ok, encoded = cv2.imencode(suffix, image)
    if not ok:
        raise IOError(f"Görüntü encode edilemedi: {path}")

    fd, temp_name = tempfile.mkstemp(
        dir=str(path.parent),
        prefix=f".{path.stem}_",
        suffix=f"{suffix}.tmp",
    )
    os.close(fd)
    temp_path = Path(temp_name)
    try:
        temp_path.write_bytes(encoded.tobytes())
        os.replace(temp_path, path)
    finally:
        temp_path.unlink(missing_ok=True)


def atomic_save_landmarks(
    path: Path,
    normalized_xyz: np.ndarray,
    pixel_xy: np.ndarray,
    image_width: int,
    image_height: int,
    left_bbox: tuple[int, int, int, int] | None,
    right_bbox: tuple[int, int, int, int] | None,
    combined_bbox: tuple[int, int, int, int] | None,
) -> None:
    """Landmark verisini sıkıştırılmış NPZ olarak atomik kaydeder."""
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(
        dir=str(path.parent), prefix=f".{path.stem}_", suffix=".npz.tmp"
    )
    os.close(fd)
    temp_path = Path(temp_name)

    def bbox_array(bbox):
        return np.asarray(bbox if bbox is not None else (-1, -1, -1, -1), dtype=np.int32)

    try:
        # np.savez_compressed uzantı eklememesi için açık dosya nesnesi kullanılır.
        with temp_path.open("wb") as f:
            np.savez_compressed(
                f,
                all_landmarks_normalized_xyz=normalized_xyz.astype(np.float32),
                all_landmarks_pixel_xy=pixel_xy.astype(np.int32),
                left_eye_indices=np.asarray(LEFT_EYE_CONTOUR, dtype=np.int16),
                right_eye_indices=np.asarray(RIGHT_EYE_CONTOUR, dtype=np.int16),
                left_iris_indices=np.asarray(LEFT_IRIS, dtype=np.int16),
                right_iris_indices=np.asarray(RIGHT_IRIS, dtype=np.int16),
                left_eye_normalized_xyz=normalized_xyz[valid_indices(pixel_xy, LEFT_EYE_CONTOUR)],
                right_eye_normalized_xyz=normalized_xyz[valid_indices(pixel_xy, RIGHT_EYE_CONTOUR)],
                left_iris_normalized_xyz=normalized_xyz[valid_indices(pixel_xy, LEFT_IRIS)],
                right_iris_normalized_xyz=normalized_xyz[valid_indices(pixel_xy, RIGHT_IRIS)],
                left_eye_pixel_xy=pixel_xy[valid_indices(pixel_xy, LEFT_EYE_CONTOUR)],
                right_eye_pixel_xy=pixel_xy[valid_indices(pixel_xy, RIGHT_EYE_CONTOUR)],
                left_iris_pixel_xy=pixel_xy[valid_indices(pixel_xy, LEFT_IRIS)],
                right_iris_pixel_xy=pixel_xy[valid_indices(pixel_xy, RIGHT_IRIS)],
                left_bbox=bbox_array(left_bbox),
                right_bbox=bbox_array(right_bbox),
                combined_bbox=bbox_array(combined_bbox),
                image_size_wh=np.asarray((image_width, image_height), dtype=np.int32),
            )
            f.flush()
            os.fsync(f.fileno())
        os.replace(temp_path, path)
    finally:
        temp_path.unlink(missing_ok=True)


def read_existing_metadata(csv_path: Path) -> tuple[list[dict], set[str]]:
    if not csv_path.exists():
        return [], set()

    rows: list[dict] = []
    completed: set[str] = set()
    with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
            if row.get("status") in {
                "ok", "eyes_not_detected", "no_face", "read_error", "detect_error"
            }:
                completed.add(row.get("relative_frame_path", ""))
    return rows, completed


def atomic_write_csv(csv_path: Path, rows: Sequence[dict]) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(
        dir=str(csv_path.parent),
        prefix=f".{csv_path.stem}_",
        suffix=".csv.tmp",
    )
    os.close(fd)
    temp_path = Path(temp_name)
    try:
        with temp_path.open("w", encoding="utf-8-sig", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=METADATA_FIELDS)
            writer.writeheader()
            for row in rows:
                writer.writerow({field: row.get(field, "") for field in METADATA_FIELDS})
            f.flush()
            os.fsync(f.fileno())
        os.replace(temp_path, csv_path)
    finally:
        temp_path.unlink(missing_ok=True)


def bbox_values(prefix: str, bbox: tuple[int, int, int, int] | None) -> dict:
    if bbox is None:
        return {
            f"{prefix}_bbox_x1": "",
            f"{prefix}_bbox_y1": "",
            f"{prefix}_bbox_x2": "",
            f"{prefix}_bbox_y2": "",
        }
    x1, y1, x2, y2 = bbox
    return {
        f"{prefix}_bbox_x1": x1,
        f"{prefix}_bbox_y1": y1,
        f"{prefix}_bbox_x2": x2,
        f"{prefix}_bbox_y2": y2,
    }


def draw_debug(
    image_bgr: np.ndarray,
    face_points: Sequence[np.ndarray],
    boxes: Sequence[tuple[
        tuple[int, int, int, int] | None,
        tuple[int, int, int, int] | None,
        tuple[int, int, int, int] | None,
    ]],
) -> np.ndarray:
    debug = image_bgr.copy()

    for face_id, (points, (left_bbox, right_bbox, combined_bbox)) in enumerate(
        zip(face_points, boxes)
    ):
        for bbox, color, label in [
            (left_bbox, (0, 255, 0), f"F{face_id} LEFT"),
            (right_bbox, (255, 150, 0), f"F{face_id} RIGHT"),
            (combined_bbox, (255, 0, 255), f"F{face_id} COMBINED"),
        ]:
            if bbox is None:
                continue
            x1, y1, x2, y2 = bbox
            cv2.rectangle(debug, (x1, y1), (x2, y2), color, 1)
            cv2.putText(
                debug,
                label,
                (x1, max(15, y1 - 4)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.38,
                color,
                1,
                cv2.LINE_AA,
            )

        # Göz ve iris landmark noktaları
        for idx in valid_indices(
            points,
            LEFT_EYE_CONTOUR + RIGHT_EYE_CONTOUR + LEFT_IRIS + RIGHT_IRIS,
        ):
            x, y = points[idx]
            cv2.circle(debug, (int(x), int(y)), 1, (0, 255, 255), -1)

    return debug


def relative_output_base(
    cfg: Config,
    frame_path: Path,
    label: str,
    split: str,
    video_id: str,
) -> Path:
    """ROI türünden sonra kullanılacak video alt yolunu döndürür.

    Ana çıktı sırası ayrıca process_frame içinde:
      label / split / left|right|combined|landmarks|debug / video_id

    şeklinde kurulmaktadır.
    """
    if label != "unknown" and split != "unknown":
        return Path(video_id)
    rel_parent = frame_path.relative_to(cfg.input_root).parent
    return Path("unknown_source") / rel_parent


def estimate_iris_visibility(
    points: np.ndarray,
    eye_indices: Sequence[int],
    iris_indices: Sequence[int],
) -> bool:
    """İris landmarklarının göz kutusu içinde ve göz yüksekliğinin yeterli olup
    olmadığına ilişkin kaba bir görünürlük tahmini üretir.

    Bu değer kalite etiketi olarak kullanılır; görüntü bu aşamada silinmez.
    """
    eye_ids = valid_indices(points, eye_indices)
    iris_ids = valid_indices(points, iris_indices)
    if len(eye_ids) < 4 or len(iris_ids) < 5:
        return False

    eye = points[eye_ids]
    iris = points[iris_ids]
    x1, y1 = eye.min(axis=0)
    x2, y2 = eye.max(axis=0)
    eye_h = int(y2 - y1)
    eye_w = int(x2 - x1)
    if eye_h < 3 or eye_w < 6:
        return False

    center = iris.mean(axis=0)
    margin_x = max(1.0, eye_w * 0.12)
    margin_y = max(1.0, eye_h * 0.35)
    return bool(
        (x1 - margin_x <= center[0] <= x2 + margin_x)
        and (y1 - margin_y <= center[1] <= y2 + margin_y)
    )


def make_base_row(
    frame_path: Path,
    cfg: Config,
    label: str,
    split: str,
    video_id: str,
) -> dict:
    relative = frame_path.relative_to(cfg.input_root).as_posix()
    return {
        "sample_id": "",
        "source_frame": str(frame_path),
        "relative_frame_path": relative,
        "label": label,
        "split": split,
        "video_id": video_id,
        "frame_stem": frame_path.stem,
        "face_id": "",
        "face_id_scope": "frame_local_detection_order",
        "faces_in_frame": 0,
        "left_eye_detected": False,
        "right_eye_detected": False,
        "left_eye_path": "",
        "right_eye_path": "",
        "combined_eye_path": "",
        "debug_path": "",
        "landmarks_path": "",
        **bbox_values("left", None),
        **bbox_values("right", None),
        **bbox_values("combined", None),
        "left_eye_width_px": "",
        "left_eye_height_px": "",
        "right_eye_width_px": "",
        "right_eye_height_px": "",
        "left_iris_landmarks_available": False,
        "right_iris_landmarks_available": False,
        "left_iris_visible_estimate": False,
        "right_iris_visible_estimate": False,
        "landmark_count": 0,
        "status": "",
        "error": "",
        "processing_ms": "",
    }


def process_frame(
    frame_path: Path,
    cfg: Config,
    landmarker,
) -> list[dict]:
    start = time.perf_counter()
    label, split, video_id = infer_dataset_fields(frame_path, cfg.input_root)
    base_row = make_base_row(frame_path, cfg, label, split, video_id)

    image_bgr = cv2.imread(str(frame_path))
    if image_bgr is None:
        base_row["status"] = "read_error"
        base_row["error"] = "cv2.imread görüntüyü okuyamadı"
        base_row["processing_ms"] = round((time.perf_counter() - start) * 1000, 2)
        return [base_row]

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

    try:
        result = landmarker.detect(mp_image)
    except Exception as exc:
        base_row["status"] = "detect_error"
        base_row["error"] = f"{type(exc).__name__}: {exc}"
        base_row["processing_ms"] = round((time.perf_counter() - start) * 1000, 2)
        return [base_row]

    faces = result.face_landmarks or []
    if not faces:
        base_row["status"] = "no_face"
        base_row["processing_ms"] = round((time.perf_counter() - start) * 1000, 2)
        return [base_row]

    height, width = image_bgr.shape[:2]
    frame_relative = frame_path.relative_to(cfg.input_root).as_posix()
    output_base = relative_output_base(cfg, frame_path, label, split, video_id)

    frame_rows: list[dict] = []
    debug_points: list[np.ndarray] = []
    debug_boxes = []

    for face_id, landmarks in enumerate(faces):
        normalized_xyz, points = landmarks_to_arrays(landmarks, width, height)

        left_bbox = bbox_from_indices(
            points, LEFT_EYE_CONTOUR + LEFT_IRIS,
            width, height, cfg.eye_padding_ratio
        )
        right_bbox = bbox_from_indices(
            points, RIGHT_EYE_CONTOUR + RIGHT_IRIS,
            width, height, cfg.eye_padding_ratio
        )
        combined_bbox = union_bbox(
            left_bbox, right_bbox, width, height, cfg.combined_padding_ratio
        )

        left_crop = crop_and_letterbox(image_bgr, left_bbox, cfg.output_size)
        right_crop = crop_and_letterbox(image_bgr, right_bbox, cfg.output_size)
        combined_crop = crop_and_letterbox(image_bgr, combined_bbox, cfg.output_size)

        filename = f"{frame_path.stem}__face_{face_id:02d}.jpg"

        split_root = cfg.output_root / label / split

        left_path = split_root / "left" / output_base / filename
        right_path = split_root / "right" / output_base / filename
        combined_path = split_root / "combined" / output_base / filename

        if cfg.save_left_right and left_crop is not None:
            atomic_imwrite(left_path, left_crop)
        if cfg.save_left_right and right_crop is not None:
            atomic_imwrite(right_path, right_crop)
        if cfg.save_combined and combined_crop is not None:
            atomic_imwrite(combined_path, combined_crop)

        landmarks_path = (
            split_root / "landmarks" / output_base /
            f"{frame_path.stem}__face_{face_id:02d}.npz"
        )
        atomic_save_landmarks(
            landmarks_path,
            normalized_xyz,
            points,
            width,
            height,
            left_bbox,
            right_bbox,
            combined_bbox,
        )

        row = make_base_row(frame_path, cfg, label, split, video_id)
        row.update({
            "sample_id": stable_sample_id(frame_relative, face_id),
            "face_id": face_id,
            "faces_in_frame": len(faces),
            "left_eye_detected": left_crop is not None,
            "right_eye_detected": right_crop is not None,
            "left_eye_path": (
                left_path.relative_to(cfg.output_root).as_posix()
                if left_crop is not None else ""
            ),
            "right_eye_path": (
                right_path.relative_to(cfg.output_root).as_posix()
                if right_crop is not None else ""
            ),
            "combined_eye_path": (
                combined_path.relative_to(cfg.output_root).as_posix()
                if combined_crop is not None else ""
            ),
            "landmarks_path": landmarks_path.relative_to(cfg.output_root).as_posix(),
            "left_eye_width_px": (
                left_bbox[2] - left_bbox[0] if left_bbox else ""
            ),
            "left_eye_height_px": (
                left_bbox[3] - left_bbox[1] if left_bbox else ""
            ),
            "right_eye_width_px": (
                right_bbox[2] - right_bbox[0] if right_bbox else ""
            ),
            "right_eye_height_px": (
                right_bbox[3] - right_bbox[1] if right_bbox else ""
            ),
            "left_iris_landmarks_available": all(
                idx < len(points) for idx in LEFT_IRIS
            ),
            "right_iris_landmarks_available": all(
                idx < len(points) for idx in RIGHT_IRIS
            ),
            "left_iris_visible_estimate": estimate_iris_visibility(
                points, LEFT_EYE_CONTOUR, LEFT_IRIS
            ),
            "right_iris_visible_estimate": estimate_iris_visibility(
                points, RIGHT_EYE_CONTOUR, RIGHT_IRIS
            ),
            "landmark_count": len(points),
            "status": (
                "ok"
                if left_crop is not None or right_crop is not None
                else "eyes_not_detected"
            ),
            **bbox_values("left", left_bbox),
            **bbox_values("right", right_bbox),
            **bbox_values("combined", combined_bbox),
        })
        frame_rows.append(row)
        debug_points.append(points)
        debug_boxes.append((left_bbox, right_bbox, combined_bbox))

    if cfg.save_debug:
        debug_image = draw_debug(image_bgr, debug_points, debug_boxes)
        split_root = cfg.output_root / label / split
        debug_path = (
            split_root / "debug" / output_base /
            f"{frame_path.stem}__debug.jpg"
        )
        atomic_imwrite(debug_path, debug_image)
        debug_rel = debug_path.relative_to(cfg.output_root).as_posix()
        for row in frame_rows:
            row["debug_path"] = debug_rel

    processing_ms = round((time.perf_counter() - start) * 1000, 2)
    for row in frame_rows:
        row["processing_ms"] = processing_ms

    return frame_rows


def create_landmarker(cfg: Config):
    BaseOptions = mp.tasks.BaseOptions
    FaceLandmarker = mp.tasks.vision.FaceLandmarker
    FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
    VisionRunningMode = mp.tasks.vision.RunningMode

    options = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(cfg.model_path)),
        running_mode=VisionRunningMode.IMAGE,
        num_faces=cfg.num_faces,
        min_face_detection_confidence=cfg.min_face_detection_confidence,
        min_face_presence_confidence=cfg.min_face_presence_confidence,
        min_tracking_confidence=cfg.min_tracking_confidence,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
    )
    return FaceLandmarker.create_from_options(options)


def save_run_config(cfg: Config) -> None:
    cfg.output_root.mkdir(parents=True, exist_ok=True)
    serializable = asdict(cfg)
    serializable = {
        key: str(value) if isinstance(value, Path) else value
        for key, value in serializable.items()
    }
    config_path = cfg.output_root / "run_config.json"
    config_path.write_text(
        json.dumps(serializable, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def main() -> int:
    cfg = parse_args()
    validate_config(cfg)
    save_run_config(cfg)

    metadata_path = cfg.output_root / "metadata.csv"
    existing_rows, completed_frames = read_existing_metadata(metadata_path)

    if cfg.overwrite:
        existing_rows = []
        completed_frames = set()

    images = iter_images(cfg.input_root)
    pending = [
        p for p in images
        if p.relative_to(cfg.input_root).as_posix() not in completed_frames
    ]

    print("=" * 72)
    print("GÖZ ROI ÇIKARMA")
    print(f"Input             : {cfg.input_root}")
    print(f"Output            : {cfg.output_root}")
    print(f"Toplam frame      : {len(images)}")
    print(f"Önceden tamamlanan: {len(completed_frames)}")
    print(f"Bu çalıştırma     : {len(pending)}")
    print(f"Maksimum yüz      : {cfg.num_faces}")
    print(f"Confidence        : det={cfg.min_face_detection_confidence}, "
          f"presence={cfg.min_face_presence_confidence}, "
          f"tracking={cfg.min_tracking_confidence}")
    print("=" * 72)

    if not pending:
        print("İşlenecek yeni frame yok.")
        return 0

    all_rows = list(existing_rows)
    processed_since_flush = 0
    start_all = time.perf_counter()

    with create_landmarker(cfg) as landmarker:
        for index, frame_path in enumerate(pending, start=1):
            rows = process_frame(frame_path, cfg, landmarker)
            all_rows.extend(rows)
            processed_since_flush += 1

            if processed_since_flush >= cfg.flush_every:
                atomic_write_csv(metadata_path, all_rows)
                processed_since_flush = 0

            if index == 1 or index % 25 == 0 or index == len(pending):
                ok_faces = sum(
                    1 for row in all_rows
                    if row.get("status") == "ok"
                )
                elapsed_min = (time.perf_counter() - start_all) / 60
                print(
                    f"[{index}/{len(pending)}] "
                    f"son={frame_path.name} | "
                    f"toplam başarılı yüz={ok_faces} | "
                    f"süre={elapsed_min:.1f} dk"
                )

    atomic_write_csv(metadata_path, all_rows)

    status_counts: dict[str, int] = {}
    for row in all_rows:
        status = str(row.get("status", "unknown"))
        status_counts[status] = status_counts.get(status, 0) + 1

    print("\nTamamlandı.")
    print(f"Metadata: {metadata_path}")
    print(f"Durumlar : {status_counts}")
    return 0





def write_split_outputs(cfg: Config, rows: Sequence[dict]) -> None:
    """Her real/fake ve train/val/test klasörüne metadata.csv ve statistics.json yazar."""
    valid_labels = ("real", "fake")
    valid_splits = ("train", "val", "test")

    for label in valid_labels:
        for split in valid_splits:
            split_rows = [
                row for row in rows
                if row.get("label") == label and row.get("split") == split
            ]
            split_root = cfg.output_root / label / split
            split_root.mkdir(parents=True, exist_ok=True)

            # İstenen alt klasörler veri olmasa bile standart olarak oluşturulur.
            for folder in ("left", "right", "combined", "landmarks", "debug"):
                (split_root / folder).mkdir(parents=True, exist_ok=True)

            atomic_write_csv(split_root / "metadata.csv", split_rows)

            unique_frames = {
                str(row.get("relative_frame_path", ""))
                for row in split_rows
                if row.get("relative_frame_path")
            }
            status_counts: dict[str, int] = {}
            for row in split_rows:
                status = str(row.get("status", "unknown"))
                status_counts[status] = status_counts.get(status, 0) + 1

            statistics = {
                "label": label,
                "split": split,
                "unique_source_frames": len(unique_frames),
                "metadata_rows": len(split_rows),
                "detected_faces": sum(
                    1 for row in split_rows if row.get("status") == "ok"
                ),
                "left_eye_outputs": sum(
                    str(row.get("left_eye_path", "")).strip() != ""
                    for row in split_rows
                ),
                "right_eye_outputs": sum(
                    str(row.get("right_eye_path", "")).strip() != ""
                    for row in split_rows
                ),
                "combined_outputs": sum(
                    str(row.get("combined_eye_path", "")).strip() != ""
                    for row in split_rows
                ),
                "multi_face_frames": len({
                    str(row.get("relative_frame_path"))
                    for row in split_rows
                    if int(row.get("faces_in_frame") or 0) > 1
                }),
                "status_counts": status_counts,
            }
            atomic_write_json(split_root / "statistics.json", statistics)


def atomic_write_json(path: Path, data: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(
        dir=str(path.parent),
        prefix=f".{path.stem}_",
        suffix=".json.tmp",
    )
    os.close(fd)
    temp_path = Path(temp_name)
    try:
        temp_path.write_text(
            json.dumps(data, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        os.replace(temp_path, path)
    finally:
        temp_path.unlink(missing_ok=True)


def run_pipeline(cfg: Config, max_frames: int | None = None) -> Path:
    """Colab uyumlu ana çalışma fonksiyonu."""
    validate_config(cfg)
    save_run_config(cfg)

    metadata_path = cfg.output_root / "metadata.csv"
    existing_rows, completed_frames = read_existing_metadata(metadata_path)

    if cfg.overwrite:
        existing_rows = []
        completed_frames = set()

    images = iter_images(cfg.input_root)
    pending = [
        p for p in images
        if p.relative_to(cfg.input_root).as_posix() not in completed_frames
    ]
    if max_frames is not None:
        pending = pending[:max_frames]

    print("=" * 72)
    print("GÖZ ROI ÇIKARMA — COLAB")
    print(f"Input             : {cfg.input_root}")
    print(f"Output            : {cfg.output_root}")
    print(f"Toplam frame      : {len(images)}")
    print(f"Önceden tamamlanan: {len(completed_frames)}")
    print(f"Bu çalıştırma     : {len(pending)}")
    print(f"Maksimum yüz      : {cfg.num_faces}")
    print("=" * 72)

    if not pending:
        print("İşlenecek yeni frame yok.")
        return metadata_path

    all_rows = list(existing_rows)
    processed_since_flush = 0
    start_all = time.perf_counter()

    try:
        with create_landmarker(cfg) as landmarker:
            for index, frame_path in enumerate(pending, start=1):
                rows = process_frame(frame_path, cfg, landmarker)
                all_rows.extend(rows)
                processed_since_flush += 1

                if processed_since_flush >= cfg.flush_every:
                    atomic_write_csv(metadata_path, all_rows)
                    write_split_outputs(cfg, all_rows)
                    processed_since_flush = 0

                if index == 1 or index % 25 == 0 or index == len(pending):
                    ok_faces = sum(1 for row in all_rows if row.get("status") == "ok")
                    elapsed_min = (time.perf_counter() - start_all) / 60
                    print(
                        f"[{index}/{len(pending)}] son={frame_path.name} | "
                        f"başarılı yüz={ok_faces} | süre={elapsed_min:.1f} dk"
                    )
    except KeyboardInterrupt:
        print("Kullanıcı durdurdu; mevcut sonuçlar CSV'ye kaydediliyor...")
    finally:
        atomic_write_csv(metadata_path, all_rows)
        write_split_outputs(cfg, all_rows)

    status_counts: dict[str, int] = {}
    for row in all_rows:
        status = str(row.get("status", "unknown"))
        status_counts[status] = status_counts.get(status, 0) + 1

    print("\nTamamlandı / checkpoint kaydedildi.")
    print(f"Metadata: {metadata_path}")
    print(f"Durumlar : {status_counts}")
    return metadata_path


In [ ]:
# 6) Yapılandırmayı oluştur ve çalıştır
cfg = Config(
    input_root=INPUT_ROOT.resolve(),
    output_root=OUTPUT_ROOT.resolve(),
    model_path=MODEL_PATH.resolve(),
    num_faces=NUM_FACES,
    min_face_detection_confidence=DET_CONF,
    min_face_presence_confidence=PRESENCE_CONF,
    min_tracking_confidence=TRACKING_CONF,
    eye_padding_ratio=EYE_PADDING,
    combined_padding_ratio=COMBINED_PADDING,
    output_size=OUTPUT_SIZE,
    save_left_right=True,
    save_combined=True,
    save_debug=SAVE_DEBUG,
    flush_every=FLUSH_EVERY,
    overwrite=OVERWRITE,
)

metadata_path = run_pipeline(cfg, max_frames=SMOKE_TEST_LIMIT)


In [ ]:
# 7) Metadata ve klasör yapısı kontrolü
import pandas as pd

metadata = pd.read_csv(metadata_path)
print('Global metadata satır sayısı:', len(metadata))
print('\nDurum dağılımı:')
display(metadata['status'].value_counts(dropna=False).rename_axis('status').reset_index(name='count'))

kontrol_kolonlari = [
    'relative_frame_path', 'label', 'split', 'video_id',
    'face_id', 'face_id_scope', 'faces_in_frame',
    'left_eye_path', 'right_eye_path', 'combined_eye_path',
    'landmarks_path', 'status', 'error'
]
display(metadata[kontrol_kolonlari].head(20))

print('\nOluşturulan ana klasörler:')
for label in ('real', 'fake'):
    for split in ('train', 'val', 'test'):
        root = OUTPUT_ROOT / label / split
        print(root.relative_to(OUTPUT_ROOT), '->', [p.name for p in root.iterdir()] if root.exists() else 'YOK')


In [ ]:
# 8) Debug görüntülerinden örnek göster
import matplotlib.pyplot as plt
import cv2

valid_debug = metadata['debug_path'].dropna().astype(str)
valid_debug = valid_debug[valid_debug.str.len() > 0].drop_duplicates().head(6)

for rel_path in valid_debug:
    img_path = OUTPUT_ROOT / rel_path
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 7))
    plt.imshow(img)
    plt.title(rel_path)
    plt.axis('off')
    plt.show()


## 9) Tam veri setini çalıştırma

Debug görüntüleri doğruysa ayar hücresinde:

```python
SMOKE_TEST_LIMIT = None
```

yapıp **6. hücreyi yeniden çalıştır**. `OVERWRITE=False` olduğu için daha önce tamamlanan frame'ler atlanır ve işlem kaldığı yerden devam eder.

In [ ]:
# 10) Çıktı muhasebesi ve temel doğrulama
assert metadata['relative_frame_path'].notna().all(), 'Boş frame yolu var.'
assert metadata['status'].notna().all(), 'Boş status var.'
assert set(metadata['label'].dropna().unique()).issubset({'real', 'fake', 'unknown'})
assert set(metadata['split'].dropna().unique()).issubset({'train', 'val', 'test', 'unknown'})

ok = metadata[metadata['status'] == 'ok']
missing_landmarks = ok['landmarks_path'].isna() | (ok['landmarks_path'].astype(str).str.len() == 0)
assert not missing_landmarks.any(), 'Başarılı satırlardan bazılarında landmark yolu yok.'

for label in ('real', 'fake'):
    for split in ('train', 'val', 'test'):
        split_root = OUTPUT_ROOT / label / split
        for folder in ('left', 'right', 'combined', 'landmarks', 'debug'):
            assert (split_root / folder).exists(), f'Eksik klasör: {split_root / folder}'
        assert (split_root / 'metadata.csv').exists(), f'Eksik metadata: {split_root}'
        assert (split_root / 'statistics.json').exists(), f'Eksik statistics: {split_root}'

print('✅ Temel kontroller geçti.')
print('Başarılı yüz sayısı:', len(ok))
print('Benzersiz kaynak frame:', metadata['relative_frame_path'].nunique())
print('Toplam metadata satırı:', len(metadata))
